# Week 11 Live Session: Generative AI — LLMs and Diffusion

## 🎓 COURSE FINALE (before Week 12 Copilot Workshop)

## What you'll do today

1. **Recap** the path: BERT (2018) → GPT-2/3 (2019-2020) → ChatGPT (2022) → today
2. **LLMs intuitively:** decoder transformers + scale + RLHF
3. **Live prompting:** load a small LLM, learn prompt engineering
4. **Diffusion intuitively:** add noise then learn to reverse it
5. **Live image generation:** Stable Diffusion via `diffusers`
6. **Multimodal tour:** CLIP, vision-language, text-to-video, audio
7. **Course wrap-up:** From linear regression to image generation in 11 weeks

## Continuity from Week 10
Last week: encoder transformers (DistilBERT) for understanding. **Today: decoder transformers (LLMs) for generation, plus diffusion for images.** Same `from_pretrained` pattern, scaled up.

## What's next
Week 12 is the Copilot Workshop. After today, you've completed the substantive ML content of the course.

## Section 0: Setup

In [ ]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, set_seed
import numpy as np

set_seed(42)

# Runs on GPU, Apple Silicon (MPS), or CPU — no code changes needed.
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
elif DEVICE == 'mps':
    print('Apple Silicon (MPS) — runs well; Phi-3-mini or Qwen-0.5B recommended.')
else:
    print('CPU only — generation is slow. Use Qwen2.5-0.5B-Instruct for class.')

## Section 1: LLMs Intuitively

### What's an LLM, really?

**An LLM is a decoder-only transformer trained on a massive corpus to predict the next token.**

That's it. The training objective is staggeringly simple:
1. Read the first N tokens of some text
2. Predict the (N+1)th token
3. Compare to the actual next token, compute loss, update weights
4. Repeat for trillions of tokens

**At small scale, this gives you a toy text predictor.** At GPT-3 scale (175B parameters, hundreds of billions of training tokens), this gives you something that can do reasoning, code, math, dialogue, translation, summarization, and tasks the trainers never explicitly designed for. **Emergent capabilities** — they appear at scale, not at small sizes.

### The training stack

Modern LLMs (ChatGPT, Claude, Llama-Chat) go through three stages:

1. **Pretraining:** Predict next token on ~10 trillion tokens of internet data. Output: smart but not aligned. Says random things, makes mistakes, occasionally toxic.

2. **Instruction tuning (Supervised Fine-Tuning, SFT):** Fine-tune on curated (instruction, response) pairs. "When the user says X, respond with Y." Output: follows instructions.

3. **RLHF (Reinforcement Learning from Human Feedback):** Humans rank model outputs from best to worst. Train a reward model on the rankings. Use RL to make the LLM produce higher-ranked outputs. Output: helpful, harmless, honest.

**Pretraining made it smart. RLHF made it helpful.**

### The open-source landscape

| Model | Params | Size | Notes |
|---|---|---|---|
| GPT-4, Claude 3+ | unknown (huge) | API only | Frontier closed-source |
| Llama 3 70B | 70B | ~140GB | Open weights from Meta |
| Mistral 7B | 7B | ~14GB | Strong small model |
| **Phi-3-mini** | **3.8B** | **~2GB** | **What we'll use today** |
| **Qwen2.5-0.5B** | **0.5B** | **~500MB** | **Smallest usable** |

## Section 2: Live Prompting

Let's load a small LLM and explore prompt engineering.

In [ ]:
# Pick based on your machine. Phi-3-mini is more capable; Qwen-0.5B is faster / lighter.
MODEL = 'microsoft/Phi-3-mini-4k-instruct'
# MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'  # alternative for slower machines / less memory

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    dtype=torch.float16 if DEVICE in ('cuda', 'mps') else torch.float32,  # v5: 'dtype' (was 'torch_dtype')
    device_map='auto' if DEVICE == 'cuda' else None,
    # NOTE: do NOT pass trust_remote_code=True — transformers v5 has native Phi-3/Qwen support,
    # and the models' old bundled code crashes on v5 (KeyError: 'type' in rope init).
)
if DEVICE != 'cuda':          # cuda is placed by device_map='auto'; mps/cpu need an explicit move
    model = model.to(DEVICE)

print(f'Loaded {MODEL}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
def generate(prompt, max_new_tokens=120, temperature=0.7, top_p=0.9):
    """Helper to generate text from the loaded LLM."""
    messages = [{'role': 'user', 'content': prompt}]
    # transformers v5: apply_chat_template returns a BatchEncoding (dict of input_ids +
    # attention_mask), NOT a bare tensor. Use return_dict=True and unpack with generate(**inputs).
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors='pt', add_generation_prompt=True, return_dict=True
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

# Quick test
print(generate('Explain transformers in two sentences.'))

### Pattern 1: Zero-shot

Just ask. The model uses its pretrained knowledge.

In [ ]:
prompt = 'Classify this customer review as positive or negative: "The product arrived broken and customer service was unhelpful."'
print(generate(prompt, max_new_tokens=20))

### Pattern 2: Few-shot

Provide examples in the prompt to guide format.

In [ ]:
prompt = '''Classify customer reviews. Examples:
Review: "Love it, works perfectly!"
Sentiment: POSITIVE

Review: "Battery died after one week."
Sentiment: NEGATIVE

Review: "It's okay, nothing special."
Sentiment: NEUTRAL

Review: "Best purchase I've made this year."
Sentiment:'''

print(generate(prompt, max_new_tokens=10))

### Pattern 3: Structured output (JSON)

Force the model to return parseable JSON.

In [ ]:
prompt = '''Extract product information from this customer message and return as JSON.

Message: "I bought the Nikon Z6 II camera last month for $1900 from B&H Photo. The serial number is ZB123456."

Return ONLY valid JSON with keys: product, brand, price, vendor, serial_number'''

print(generate(prompt, max_new_tokens=100, temperature=0.1))  # low temp for deterministic output

### Pattern 4: Temperature effects

Same prompt, different temperatures.

In [ ]:
prompt = 'Write a one-sentence tagline for a coffee shop:'

for temp in [0.0, 0.5, 1.0]:
    print(f'\nTemperature={temp}:')
    print(f'  {generate(prompt, max_new_tokens=30, temperature=temp)}')

**Observation:**
- **temp=0** → deterministic, often boring
- **temp=0.5-0.7** → creative but coherent (most common setting)
- **temp=1.0+** → creative but may go off-track

### LLM caveats — IMPORTANT

**LLMs hallucinate.** They generate confident-sounding text even when wrong. Test:

In [ ]:
# Ask about something fictional or rare and watch it confidently invent
prompt = 'Tell me about the historical event that occurred in Smithville, Pennsylvania on March 12, 1843.'
print(generate(prompt, max_new_tokens=100, temperature=0.3))
print('\n[Likely confidently invented. There may be no such event.]')
print('[For factual tasks, use retrieval-augmented generation (RAG) and verify outputs.]')

## Section 3: Diffusion Intuitively

### How diffusion models work

Diffusion is a totally different paradigm from transformers. The training is a clever trick:

**Forward process (no learning required):**
- Start with a real image
- Add a small amount of Gaussian noise
- Repeat T times (typically T=1000)
- After T steps, the image is pure noise

Anyone can do this — it's just adding random noise.

**Reverse process (this is where the model learns):**
- Train a neural network (usually a UNet) to undo ONE step of noise at a time
- Given a slightly noisy image, predict the slightly less noisy version
- The model never sees the full reverse process — it only learns one step

**Sampling (generating new images):**
- Start with pure random noise
- Apply the trained reverse step ~50 times
- Out comes a coherent, realistic image

### Latent diffusion (Stable Diffusion)

Pure-pixel diffusion is slow (millions of pixels per image, 50 forward passes). Stable Diffusion runs the diffusion in a compressed latent space (~64×64×4 = 16K dimensions), then decodes to full resolution. **This is what makes Stable Diffusion fast enough to run on consumer hardware.**

### Text conditioning

How does "a cat wearing a top hat" become an image of a cat wearing a top hat? **Cross-attention.** The text is encoded by CLIP (or similar) into embeddings. During the reverse diffusion process, the UNet's denoising is conditioned on these text embeddings via cross-attention layers.

## Section 4: Live Image Generation

In [ ]:
# Free the LLM before loading Stable Diffusion (important on MPS/CPU — limited unified memory)
import gc
del model
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
elif DEVICE == 'mps':
    torch.mps.empty_cache()

from diffusers import StableDiffusionPipeline   # needs: pip install diffusers

# 'sd-legacy' is Hugging Face's maintained mirror of the original Stable Diffusion v1.5 (~4GB).
# NOTE: diffusers still uses `torch_dtype` (transformers v5 renamed it to `dtype`, diffusers did not).
# float16 only on CUDA — on MPS/CPU, fp16 Stable Diffusion can produce black images, so use float32.
pipe = StableDiffusionPipeline.from_pretrained(
    'sd-legacy/stable-diffusion-v1-5',
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
).to(DEVICE)

pipe.safety_checker = None             # disable for speed (adds latency)
if DEVICE == 'mps':
    pipe.enable_attention_slicing()    # smooths MPS memory spikes

print('Stable Diffusion loaded')

In [ ]:
# Generate one image — first try, basic prompt
import matplotlib.pyplot as plt

prompt = 'a cat wearing a top hat, photorealistic'
image = pipe(prompt, num_inference_steps=20, guidance_scale=7.5).images[0]

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title(f'Prompt: {prompt}')
plt.axis('off'); plt.show()

### Prompt engineering for images

Same prompt, additional descriptive terms changes output dramatically.

In [ ]:
prompts = [
    'a futuristic city',
    'a futuristic city, cyberpunk style, neon lights, rain, photorealistic, 8k',
    'a futuristic city, watercolor painting, soft pastels, dreamy',
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, prompt in zip(axes, prompts):
    image = pipe(prompt, num_inference_steps=20, guidance_scale=7.5).images[0]
    ax.imshow(image)
    ax.set_title(prompt[:50] + ('...' if len(prompt) > 50 else ''), fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

### Negative prompts

Specify what you DON'T want.

In [ ]:
prompt = 'a beautiful landscape, mountains and lake'
negative_prompt = 'people, buildings, cars, watermark, text'

image = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=20,
    guidance_scale=7.5
).images[0]

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title(f'+ {prompt}\n- {negative_prompt}', fontsize=9)
plt.axis('off'); plt.show()

## Section 5: Multimodal Tour (concept-only)

What's possible NOW that wasn't possible in 2021:

### CLIP — Image + Text in shared space
- Maps images and text to the same vector space
- Enables zero-shot classification of images using ANY text labels
- **Powers text-to-image:** Stable Diffusion uses CLIP to interpret your prompt
- Available: `openai/clip-vit-base-patch32` on Hugging Face

### Vision-Language Models (VLMs)
- Models that can take both image AND text input, output text
- Examples: GPT-4V, Claude (with vision), LLaVA, Qwen-VL
- Use case: "What's in this image?", "Read the receipt and total it", visual question answering

### Text-to-video
- Sora (OpenAI), Veo (Google) — generate short videos from text prompts
- Foundation: same diffusion idea, applied to video tensors instead of images

### Audio
- Whisper (OpenAI) — speech-to-text, multilingual, free, very good
- Suno, ElevenLabs — text-to-speech, music generation

### What this means for you
- For your problem, ALWAYS check if a foundation model already solves it
- Start with `pipeline()` from Hugging Face — try the easy path first
- Fine-tune or train from scratch only when foundation models genuinely don't work

## Section 6: Course Wrap-Up

### The course arc — from Week 1 to Week 11

| Week | What you built |
|---|---|
| 1 | Linear regression — find the best line through points |
| 2 | Logistic regression — binary classification |
| 3 | Trees, Random Forests, XGBoost — tabular data workhorses |
| 4 | Cross-validation, hyperparameter tuning, regularization |
| 5 | K-Means, PCA — unsupervised learning |
| 6 | First neural network with Keras |
| 7 | Dropout, EarlyStopping, model persistence |
| 8 | Convolutional neural networks |
| 9 | Transfer learning + modern CV |
| 10 | Transformers + Hugging Face |
| **11** | **LLMs + Diffusion (TODAY)** |

### What's the same across all 11 weeks?

**It's all gradient descent on a loss function.**

- Week 1: minimize MSE (squared error) by adjusting line parameters
- Week 6: minimize cross-entropy by adjusting MLP weights
- Week 8: minimize cross-entropy by adjusting CNN weights
- Week 10: minimize cross-entropy by adjusting transformer weights
- Week 11: same — LLMs minimize cross-entropy on next-token prediction; diffusion models minimize denoising loss

**The differences are scale, architecture, and 13 years of research.**

### What you can now do

- Choose the right algorithm for a problem (sklearn for tabular, CNN for images, transformers for sequences, generative models for creation)
- Implement end-to-end ML pipelines with proper train/val/test discipline
- Use pretrained models — the production default
- Fine-tune for your specific task
- Read modern ML papers and follow conferences
- Build POCs combining traditional ML + generative AI

### What's next: Week 12

Github Copilot Workshop. You'll take everything you've learned and apply it with AI-assisted coding tools. After that — you're on your own. **Go build things.**

## Final code: combining ML models

Quick demo of combining a sentiment classifier (Week 10) with an LLM response generator (Week 11). This is the kind of pipeline you can build at work.

In [ ]:
# (Conceptual — would need both models loaded)
# from transformers import pipeline
# sentiment = pipeline('sentiment-analysis')
# llm = ... (the Phi-3 we loaded earlier)

# def respond_to_customer(message):
#     # Step 1: classify sentiment
#     mood = sentiment(message)[0]['label']
#     # Step 2: generate appropriate response
#     if mood == 'NEGATIVE':
#         prompt = f'Write a sympathetic apology to this complaint: {message}'
#     else:
#         prompt = f'Write a warm thank-you to this positive feedback: {message}'
#     return llm_generate(prompt)

# This is the kind of multi-model pipeline that's now trivial to build.
print('Course content complete.')
print('See you in Week 12 for the Copilot Workshop.')
print()
print('Thank you for the journey from linear regression to image generation.')